This notebook reproduces the main findings of Kendall, Nannicini, and Trebbi's 2015 paper, which looks at how various political campaign messages affect voting. Instead of using Stata, this project uses Python with `pandas` and `statsmodels` to do the same tasks: the data handling, balance checks, and econometric modeling.

First, we bring in all necessary libraries and create a helper function. This lets us import Stata variable labels directly into a pandas DataFrame, making it easier to work with the data.

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from stargazer.stargazer import Stargazer

In [2]:
def get_labels(data):
    stata_reader = pd.read_stata(data, iterator=True)
    labels_dict = stata_reader.variable_labels()
    data_dictionary = pd.DataFrame(columns=["Variable", "Label"], data=list(labels_dict.items()))
    return data_dictionary

We're working with two main datasets_
- **Aggregate Data**, which has precinct-level info on voting results and demographic stats. 
- **Individual Data**, which includes survey responses gathered from a bunch of eligible voters before and after the election.

So, let's go ahead and load both of these datasets along with their variable labels.

In [3]:
aggregate = pd.read_stata("../Data/Kendall/aggregate.dta") # precint-level
aggregate_labels = get_labels("../Data/Kendall/aggregate.dta")

individual = pd.read_stata("../Data/Kendall/individual.dta") # individual-levels survey
individual_labels = get_labels("../Data/Kendall/individual.dta")




# Data Exploration and Balance Tests

### Aggregate Data Exploration

In [4]:
# Observations (95)
aggregate.shape

(95, 39)

In [5]:
# Variabel labels
aggregate_labels

,Variable,Label
0,building,uniquely identifies each polling site
1,college,share of college graduates
2,dummyA,valence group
3,dummyAm,valence group (mailing only)
4,dummyAp,valence group (mailing & phone)
5,dummyB,ideology group
6,dummyBm,ideology group (mailing only)
7,dummyBp,ideology group (mailing & phone)
8,dummyC,valence & ideology group
9,dummyCm,valence & ideology group (mailing only)


In [6]:
# Data types
aggregate.info()

<class 'pandas.DataFrame'>
RangeIndex: 95 entries, 0 to 94
Data columns (total 39 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   building                95 non-null     int8   
 1   college                 95 non-null     float32
 2   dummyA                  95 non-null     int8   
 3   dummyAm                 95 non-null     int8   
 4   dummyAp                 95 non-null     int8   
 5   dummyB                  95 non-null     int8   
 6   dummyBm                 95 non-null     int8   
 7   dummyBp                 95 non-null     int8   
 8   dummyC                  95 non-null     int8   
 9   dummyCm                 95 non-null     int8   
 10  dummyCp                 95 non-null     int8   
 11  dummyD                  95 non-null     int8   
 12  emp_rate                95 non-null     float32
 13  eur09_pleft             86 non-null     float32
 14  eur09_pright            86 non-null     float32
 15  eu

In [7]:
# Descriptive Statistis
aggregate.describe()

,building,college,dummyA,dummyAm,dummyAp,dummyB,dummyBm,dummyBp,dummyC,dummyCm,...,nat08_pleft,nat08_pright,nat08_turnout,reg10_pleft_cand,reg10_pright_cand,reg10_turnout_cand,saione,section,sh_houses,unemp_rate
count,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,...,84.000000,84.000000,84.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000,95.000000
mean,15.978947,1.598256,0.252632,0.126316,0.126316,0.252632,0.126316,0.126316,0.252632,0.126316,...,0.494095,0.452581,0.825329,0.292195,0.283710,0.591565,0.368421,48.000000,0.690386,0.037302
std,9.978678,1.913397,0.436827,0.333967,0.333967,0.436827,0.333967,0.333967,0.436827,0.333967,...,0.059358,0.058970,0.034997,0.044052,0.045481,0.044618,0.484935,27.568098,0.086499,0.013351
min,1.000000,0.001062,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.345747,0.329658,0.579832,0.166389,0.163265,0.346939,0.000000,1.000000,0.414615,0.000000
25%,9.000000,0.576378,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.461618,0.410092,0.813379,0.264769,0.252742,0.570675,0.000000,24.500000,0.631612,0.029328
50%,14.000000,0.986874,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.496748,0.450863,0.827316,0.293255,0.273598,0.596311,0.000000,48.000000,0.715589,0.036793
75%,23.500000,1.712082,0.500000,0.000000,0.000000,0.500000,0.000000,0.000000,0.500000,0.000000,...,0.537242,0.484627,0.842530,0.318064,0.308373,0.616529,1.000000,71.500000,0.754168,0.047678
max,38.000000,10.481249,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,0.628548,0.607547,0.885676,0.399497,0.432464,0.687204,1.000000,95.000000,0.845214,0.072887


### Ex-Ante Balancing Tests (Precinct Level) - Replicating Table A1
To check the treatment effects, we first need to make sure the randomization of campaign messages worked. Proper randomization means initial precinct traits are even between treatment and control groups.

We run tests on this by using regressions of those precinct features against the treatment dummy variables. To deal with any spatial linkages, we cluster our standard errors at the polling place, or `building`, level.

In [8]:
models = []
dep_vars = ["mun11_tenrolled", "giovi", "fiorentina", "saione", "giotto"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm"]
for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ +{'+'.join(regressors)}", data=aggregate,).fit(cov_type="cluster",
                                                                                  cov_kwds={"groups" : aggregate["building"]})
    models.append(reg)

In [9]:
table_A1 = Stargazer(models)

labels = ["Valence by Phone", "Valence by mail", "Ideology by phone", "Ideology by mail", "Double by phone", "Double by mail"]
table_A1.rename_covariates(dict(zip(regressors, labels)))
table_A1.custom_columns(["Eligible Voters", "First neighborhood","Second neighborhood", "Third neighborhood", "Fourth neighborhood" ])
table_A1.show_model_numbers(False)
table_A1.title("Ex-Ante Balancing Tests at the Precinct Level")
table_A1.covariate_order(regressors)
table_A1.show_adj_r2 = False
table_A1.show_confidence_intervals(False)
table_A1.show_degrees_of_freedom(False)
table_A1.show_f_statistic = False
table_A1.show_residual_std_err = False
table_A1

The balancing tests confirm that the predetermined variables are well-balanced across the different treatment groups. We can now transition to the individual-level survey data to perform a similar exploration and validation.

### Individual Survey Data Exploration
This dataset contains self-reported characteristics, prior beliefs, and voting declarations from the surveyed voters.

In [10]:
individual_labels

,Variable,Label
0,avg_p_a,average posterior on Fanfani's ideology from m...
1,avg_p_b,average posterior on Sestini's ideology from m...
2,avg_v_a,average posterior on Fanfani's valence from model
3,avg_v_b,average posterior on Sestini's valence from model
4,building,uniquely identifies each polling site
5,centerleft,voted for centerleft coalition in past municip...
6,college,equals 1 if university
7,dummyA,valence group
8,dummyAm,valence group (mailing only)
9,dummyAp,valence group (mailing & phone)


In [11]:
individual.shape

(1455, 59)

In [12]:
individual.dtypes

avg_p_a                      float32
avg_p_b                      float32
avg_v_a                      float32
avg_v_b                      float32
building                        int8
centerleft                      int8
college                         int8
dummyA                          int8
dummyAm                         int8
dummyAp                         int8
dummyB                          int8
dummyBm                         int8
dummyBp                         int8
dummyC                          int8
dummyCm                         int8
dummyCp                         int8
dummyD                          int8
house                           int8
other                           int8
outlf                           int8
over65                          int8
press                           int8
s1_date1                         str
s1_gender                   category
s1_ideology                 category
s1_ideology_fanfani         category
s1_ideology_fanfani_unc     category
s

In [13]:
individual.describe()

,avg_p_a,avg_p_b,avg_v_a,avg_v_b,building,centerleft,college,dummyA,dummyAm,dummyAp,...,section,std_p_a,std_p_b,std_v_a,std_v_b,turnout,tv,vote_f,vote_party_f,white
count,1306.000000,1306.000000,1306.000000,1306.000000,1455.000000,1455.000000,1455.000000,1455.000000,1455.000000,1455.000000,...,1455.000000,1306.000000,1306.000000,1306.000000,1306.000000,1455.000000,1455.000000,1306.000000,1306.000000,1455.000000
mean,2.353668,3.798451,6.936287,5.543113,14.665292,0.572509,0.192440,0.233677,0.098969,0.134708,...,44.854296,0.759310,0.726207,1.087020,1.840364,0.897595,0.703093,0.571210,0.493109,0.175945
std,0.626514,0.737652,1.493834,1.180653,9.433398,0.494885,0.394352,0.423315,0.298723,0.341529,...,27.584223,0.590506,0.632648,1.113103,1.248181,0.303285,0.457052,0.495093,0.500144,0.380904
min,1.000000,1.909500,1.000000,1.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,2.000000,3.000000,6.000000,5.500000,7.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,21.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
50%,2.371800,4.000000,7.000000,5.500000,13.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,44.000000,0.988990,0.889000,1.911800,2.872300,1.000000,1.000000,1.000000,0.000000,0.000000
75%,3.000000,4.090500,8.000000,6.000000,22.000000,1.000000,0.000000,0.000000,0.000000,0.000000,...,69.500000,1.414200,1.414200,1.961500,2.872300,1.000000,1.000000,1.000000,1.000000,0.000000
max,5.000000,5.000000,10.000000,10.000000,38.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,95.000000,1.416000,1.416000,2.889400,2.889400,1.000000,1.000000,1.000000,1.000000,1.000000


In [14]:
individual.vote_f.sum() # 746 people voted

np.float64(746.0)

### Ex-Ante Balancing Tests (Individual Level) - Replicating Table A2
Just like with the aggregate data, we must check that individual traits such as gender, age, education, and political leaning are evenly spread in the randomized groups. Standard errors are clustered at the electoral section (or precinct) level here.

In [15]:
individual["male"] = np.where(individual["s1_gender"] == "female",0,1)
dep_vars = ["male", "over65", "college", "outlf", "white", "other", "centerleft", "house", "press", "tv"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm", "C(s1_date1)"]
models = []

for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=individual).fit(cov_type="cluster",
                                                                                  cov_kwds={"groups" : individual["section"]})
    models.append(reg)
models

In [16]:
table_A2 = Stargazer(models)

table_A2.rename_covariates(dict(zip(regressors, labels)))
table_A2.custom_columns(["Male", "Over65", "College Graduate", "Out of labor force", "White collar", "Other occupation", "Center-left", "Home owner", "Read the press", "Watch TV"])
table_A2.show_model_numbers(False)
table_A2.title("Ex-Ante Balancing Tests at the Individual Level")
table_A2.covariate_order(regressors[:6])
table_A2.show_adj_r2 = False
table_A2.show_confidence_intervals(False)
table_A2.show_degrees_of_freedom(False)
table_A2.show_f_statistic = False
table_A2.show_residual_std_err = False
table_A2

## Main Results: Aggregate Treatment Effects (Table 3)

After validating the randomization, we determine the campaign messages' overall causal effects on voting. To do this, we run a regression using precinct-level results like turnout and incumbent shares. We use dummy variables for different treatments, with no message being the control. For accuracy, we cluster standard errors at the polling place level, as explained in the paper's method.

In [17]:
dep_vars = ["mun11_turnout", "mun11_pfanfani_cand", "mun11_pfanfani_parties"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm"]
models = []

for var in dep_vars:
    reg = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=aggregate).fit(cov_type="cluster",
                                                                                 cov_kwds={"groups" : aggregate["building"]})
    models.append(reg)
models


In [18]:
table3 = Stargazer(models)

table3.rename_covariates(dict(zip(regressors, labels)))
table3.custom_columns(["Turnout", "Incumber share", "Incumbent parties"])
table3.show_model_numbers(False)
table3.title("Reduced-Form Individual Estimates, All Groups")
table3.covariate_order(regressors[:6])
table3.show_adj_r2 = False
table3.show_confidence_intervals(False)
table3.show_degrees_of_freedom(False)
table3.show_f_statistic = False
table3.show_residual_std_err = False
table3

The aggregate results indicate that the **valence message delivered via phone** had a significant positive impact on the incumbent's vote share. 

## Main Results: Individual-Level Evidence (Table 4)

We run the same estimates on individual-level survey data to validate our findings. The dependent variables are binary – self-reported turnout and vote choice. So, we use a linear probability model (OLS), clustered by precinct, to stay consistent with our aggregate framework. Plus, we include fixed effects for the survey date.

In [19]:
dep_vars = ["turnout", "vote_f", "vote_party_f"]
regressors = ["dummyAp", "dummyAm", "dummyBp", "dummyBm", "dummyCp", "dummyCm", "C(s1_date1)"]
models = []

for var in dep_vars:
    mod = smf.ols(formula=f"{var} ~ {'+'.join(regressors)}", data=individual)
    reg = mod.fit(cov_type="cluster", cov_kwds={"groups" : individual.loc[mod.data.row_labels, "section"]})
    models.append(reg)
                                                                                
models

In [20]:
table3 = Stargazer(models)

table3.rename_covariates(dict(zip(regressors, labels)))
table3.custom_columns(["Turnout", "Incumber share", "Incumbent parties"])
table3.show_model_numbers(False)
table3.title("Reduced-Form Aggregate Estimates, All Groups")
table3.covariate_order(regressors[:6])
table3.show_adj_r2 = False
table3.show_confidence_intervals(False)
table3.show_degrees_of_freedom(False)
table3.show_f_statistic = False
table3.show_residual_std_err = False
table3

The individual-level estimates are coherent with the aggregate data: **phone calls focusing on the candidate's valence (competence and effort)** proved to be the most effective persuasive campaign tool, significantly increasing the incumbent's vote share. Conversely, ideological messages and messages delivered solely via mail showed negligible or statistically insignificant effects on voting choices.